# Agency Calculus: Multi-Agent Economic Simulation

> **Research-grade ABM testing how objective functions shape inequality, exploitation, trust, innovation, and sustainability.**

### Systems active
- **Sustainable Capitalism**: firms optimize min(S,E,V,C) not raw profit
- **Trust**: public reputation scores on all agents and institutions
- **Innovation**: firm R&D, tech diffusion, creative destruction
- **Horizon Index**: sustainability gate on planner objectives
- **Information/Epistemic Health**: news firms, cartel capture, R0
- **Banking**: deposits, loans, bank runs, credit cycles

### Comparison experiment
| Objective | Formula | What it tests |
|-----------|---------|---------------|
| **SUM_RAW** | `sum(wealth)` | Baseline: no sustainability constraint |
| **NASH_MIN** | `min(norm_nash, HI) * ideal` | Emergency brake on Nash welfare |
| **TOPO_X** | `topo_base * HI` | Topology shaping with proportional HI discount |
| **TOPO_MIN** | `min(norm_topo, HI) * ideal` | Topology shaping with HI emergency brake |

### Hardware
1. **Runtime > Change runtime type** > select **H100 / A100 / L4 / T4 / TPU**
2. **Runtime > Run all**

| Hardware | Backend | Parallel workers | Est. time (12 episodes x 3000 steps) |
|----------|---------|-----------------|--------------------------------------|
| H100 GPU | JAX GPU | 16 | ~30 min |
| A100 GPU | JAX GPU | 14 | ~40 min |
| L4 GPU   | JAX GPU | 10 | ~60 min |
| T4 GPU   | Numba CUDA | 6 | ~90 min |
| TPU v2   | JAX TPU | 8 | ~50 min |
| CPU only | NumPy | n_cores | ~3-5 hours |

---
## Step 1 -- Hardware detection & package installation

In [ ]:
import subprocess, sys, os

def _detect_gpu_tier():
    # Check TPU first
    try:
        if 'COLAB_TPU_ADDR' in os.environ or os.path.exists('/dev/accel0'):
            return 'tpu', 'TPU v2/v3', 0
    except Exception:
        pass
    # Check GPU
    try:
        r = subprocess.run(
            ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'],
            capture_output=True, text=True, timeout=10)
        if r.returncode == 0 and r.stdout.strip():
            line = r.stdout.strip().splitlines()[0]
            name, mem = line.split(',')[0].strip(), int(line.split(',')[1].strip())
            name_l = name.lower()
            if 'h100' in name_l: return 'h100', name, mem
            if 'a100' in name_l: return 'a100', name, mem
            if 'l4'   in name_l: return 'l4',   name, mem
            if 'l40'  in name_l: return 'l40',  name, mem
            if 't4'   in name_l: return 't4',   name, mem
            if 'v100' in name_l: return 'v100', name, mem
            return 'generic_gpu', name, mem
    except Exception:
        pass
    return 'cpu', 'none', 0

GPU_TIER, GPU_NAME, GPU_MEM_MB = _detect_gpu_tier()
print(f'Hardware  : {GPU_TIER}')
print(f'Device    : {GPU_NAME}')
if GPU_MEM_MB:
    print(f'VRAM      : {GPU_MEM_MB/1024:.1f} GB')
print(f'CPU cores : {os.cpu_count()}')

INSTALL_JAX   = GPU_TIER in ('h100', 'a100', 'l4', 'l40', 'v100', 'tpu', 'generic_gpu')
INSTALL_NUMBA = GPU_TIER == 't4'
print(f'\nBackend plan: {"JAX" if INSTALL_JAX else "Numba" if INSTALL_NUMBA else "NumPy"}')

In [ ]:
%%capture install_out
# Core dependencies
!pip install -q mesa numpy pandas scipy matplotlib seaborn networkx pyarrow tqdm psutil

# Acceleration backend
import subprocess, sys
if INSTALL_JAX:
    if GPU_TIER == 'tpu':
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        'jax[tpu]', '-f',
                        'https://storage.googleapis.com/jax-releases/libtpu_releases.html'],
                       check=False)
    else:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        'jax[cuda12]', '-f',
                        'https://storage.googleapis.com/jax-releases/jax_cuda_releases.html'],
                       check=False)
elif INSTALL_NUMBA:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numba'], check=False)

print('Installation complete.')

---
## Step 2 -- Clone repository

In [ ]:
REPO_URL = 'https://github.com/caseymrobbins/ac_sugarsims.git'
REPO_DIR = '/content/ac_sugarsims'

if os.path.isdir(REPO_DIR):
    print('Repo exists, pulling latest ...')
    !git -C {REPO_DIR} pull --rebase
else:
    print('Cloning ...')
    !git clone {REPO_URL} {REPO_DIR}

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())
!ls *.py | head -20

---
## Step 3 -- Hardware report & backend warm-up

In [ ]:
import importlib, hardware
importlib.reload(hardware)

hw  = hardware.detect_hardware()
cfg = hardware.get_accel_config(hw)

# Override worker count for known high-end GPUs
if GPU_TIER == 'h100':
    cfg.n_workers = 16
    cfg.recommended_seeds = 20
    cfg.recommended_steps = 3000
elif GPU_TIER == 'tpu':
    cfg.n_workers = 8
    cfg.recommended_seeds = 20
    cfg.recommended_steps = 3000
elif GPU_TIER == 'l40':
    cfg.n_workers = 12
    cfg.recommended_seeds = 20
    cfg.recommended_steps = 1500

print(hardware._hardware_report(hw, cfg))

# Warm up the regen function
import numpy as np
regen_fn = hardware.build_regen_fn(cfg.backend)
_f = np.ones((10, 10)) * 10.0
_r = np.ones((10, 10)) * 5.0
_l = np.ones((10, 10)) * 3.0
_w = np.ones((10, 10)) * 8.0
_p = np.zeros((10, 10))
regen_fn(_f, _r, _l, _w, _p)
print('Backend ready:', cfg.backend)

---
## Step 4 -- Smoke test (all new systems)

In [ ]:
import importlib
import environment, agents, economy, planner, metrics
import trust, innovation, sustainable_capitalism, horizon_index, banking, information
for mod in [environment, agents, economy, planner, metrics, trust, innovation,
            sustainable_capitalism, horizon_index, banking, information]:
    importlib.reload(mod)

from environment import EconomicModel

m = EconomicModel(seed=42, grid_width=20, grid_height=20,
                  n_workers=30, n_firms=3, n_landowners=2,
                  objective='TOPO_MIN', accel_config=cfg)
m._collect_animation = False
for _ in range(10):
    m.step()

last = m.metrics_history[-1]
print('Smoke test PASSED')
def _fmt(val, spec='.3f'):
    try: return f'{float(val):{spec}}'
    except (ValueError, TypeError): return str(val)
print(f'  Workers       : {len(m.workers)}')
print(f'  Gini          : {_fmt(last.get("all_gini", "?"))}')
print(f'  Agency floor  : {_fmt(last.get("agency_floor", "?"), ".4f")}')
print(f'  Horizon Index : {_fmt(last.get("horizon_index", "?"))}')
print(f'  Planner Trust : {_fmt(last.get("trust_planner", "?"))}')
print(f'  Inst. Trust   : {_fmt(last.get("trust_institutional", "?"))}')
print(f'  Tech Frontier : {_fmt(last.get("tech_frontier", "not loaded"))}')
print(f'  Firm Floor    : {_fmt(last.get("mean_firm_floor", "not loaded"))}')
print(f'  Objective     : TOPO_MIN = {m.planner.last_objective_value:.4f}')
print(f'  Backend       : {cfg.backend}')
if 'tech_frontier' not in last:
    print('\n  WARNING: Innovation metrics missing. Check that innovation.py')
    print('  is in the repo and agents.py imports from it.')
if 'mean_firm_floor' not in last:
    print('\n  WARNING: Firm floor metrics missing. Check that the updated')
    print('  metrics.py is in the repo.')

---
## Step 5 -- Comparison experiment

**4 objectives x 3 seeds x 3000 steps = 12 episodes**

| Objective | HI integration | What it tests |
|-----------|---------------|---------------|
| SUM_RAW | None | Baseline: does extraction still emerge with trust/innovation active? |
| NASH_MIN | min gate | Emergency brake: does min force faster trajectory correction? |
| TOPO_X | multiplier | Proportional discount: gentle sustainability pressure |
| TOPO_MIN | min gate | Emergency brake on topology: plan for future unless emergency |

In [ ]:
# ── Experiment configuration ──
OBJECTIVES = ['SUM_RAW', 'NASH_MIN', 'TOPO_X', 'TOPO_MIN']
SEEDS      = [42, 137, 2024]
N_STEPS    = 3000
GRID_SIZE  = 80
N_WORKERS  = 400
N_FIRMS    = 20
N_LANDOWNERS = 15
ANIMATE_SEED = 42  # collect animation for this seed only

OUTPUT_DIR = f'{REPO_DIR}/results/comparison'
os.makedirs(f'{OUTPUT_DIR}/raw_data', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/summary', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/animations', exist_ok=True)

print(f'Objectives : {OBJECTIVES}')
print(f'Seeds      : {SEEDS}')
print(f'Steps      : {N_STEPS}')
print(f'Episodes   : {len(OBJECTIVES) * len(SEEDS)}')
print(f'Grid       : {GRID_SIZE}x{GRID_SIZE}')
print(f'Agents     : {N_WORKERS} workers, {N_FIRMS} firms, {N_LANDOWNERS} landowners')

In [ ]:
import time, traceback
import pandas as pd
from metrics import episode_summary

all_summaries = []
total_t0 = time.time()

for objective in OBJECTIVES:
    print(f'\n{"=" * 55}')
    print(f'  {objective}')
    print(f'{"=" * 55}')

    for seed in SEEDS:
        try:
            animate = (seed == ANIMATE_SEED)
            print(f'  [{objective}] seed={seed} ...', end=' ', flush=True)
            t0 = time.time()

            model = EconomicModel(
                seed=seed,
                grid_width=GRID_SIZE, grid_height=GRID_SIZE,
                n_workers=N_WORKERS, n_firms=N_FIRMS,
                n_landowners=N_LANDOWNERS,
                objective=objective, accel_config=cfg,
            )
            model._collect_animation = animate

            for step in range(N_STEPS):
                model.step()
                if (step + 1) % 500 == 0:
                    print(f'{step+1}', end=' ', flush=True)

            elapsed = time.time() - t0
            print(f'done ({elapsed:.1f}s)')

            # Summary
            summary = episode_summary(model.metrics_history)
            summary['objective'] = objective
            summary['seed'] = seed
            summary['n_steps'] = N_STEPS
            summary['elapsed_seconds'] = elapsed
            all_summaries.append(summary)

            # Save raw metrics
            mdf = pd.DataFrame(model.metrics_history)
            mdf['objective'] = objective
            mdf['seed'] = seed
            mdf.to_parquet(f'{OUTPUT_DIR}/raw_data/{objective}_seed{seed}.parquet', index=False)

            # Save animation
            if animate and model.animation_frames:
                try:
                    from animate import generate_animation_html
                    generate_animation_html(
                        model.animation_frames,
                        output_path=f'{OUTPUT_DIR}/animations/{objective}_seed{seed}.html',
                        grid_size=GRID_SIZE,
                        title=f'{objective} (seed={seed})',
                        subsample=3,
                    )
                    print(f'    Animation: {OUTPUT_DIR}/animations/{objective}_seed{seed}.html')
                except Exception as e:
                    print(f'    Animation failed: {e}')

            # Free memory
            del model
            import gc; gc.collect()

        except Exception as e:
            print(f'ERROR: {e}')
            traceback.print_exc()

total_elapsed = time.time() - total_t0
print(f'\n{"=" * 55}')
print(f'  Total: {total_elapsed:.0f}s ({total_elapsed/60:.1f} min)')
print(f'  Episodes: {len(all_summaries)}/{len(OBJECTIVES)*len(SEEDS)}')
print(f'{"=" * 55}')

# Save summaries
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(f'{OUTPUT_DIR}/summary/all_summaries.csv', index=False)
summary_df.to_parquet(f'{OUTPUT_DIR}/summary/all_summaries.parquet', index=False)
print('Summaries saved.')

---
## Step 6 -- Results comparison

In [ ]:
import numpy as np

objectives = sorted(summary_df['objective'].unique())

key_metrics = [
    ('all_gini',            'Gini',             '.3f'),
    ('worker_min',          'Floor Wealth',     '.1f'),
    ('worker_mean',         'Mean Wealth',      '.1f'),
    ('agency_floor',        'Agency Floor',     '.2f'),
    ('unemployment_rate',   'Unemployment',     '.1%'),
    ('horizon_index',       'Horizon Index',    '.3f'),
    ('mean_firm_floor',     'Firm SEVC Floor',  '.3f'),
    ('trust_planner',       'Planner Trust',    '.3f'),
    ('trust_institutional', 'Inst. Trust',      '.3f'),
    ('trust_firm_mean',     'Firm Trust',       '.3f'),
    ('tech_frontier',       'Tech Frontier',    '.2f'),
    ('tech_mean',           'Tech Mean',        '.2f'),
    ('tech_gini',           'Tech Gini',        '.3f'),
    ('frac_monopoly',       '% Monopoly',       '.1%'),
    ('frac_poverty_trap',   '% Poverty Trap',   '.1%'),
    ('frac_cartel',         '% Cartel',         '.1%'),
    ('n_firms',             'Active Firms',     '.1f'),
    ('n_workers',           'Population',       '.0f'),
    ('epistemic_health',    'Epistemic Health',  '.3f'),
    ('total_production',    'Production',       '.0f'),
    ('total_debt',          'Total Debt',       '.0f'),
]

print(f'\n{"=" * 75}')
print(f'  COMPARISON: mean across {len(SEEDS)} seeds, {N_STEPS} steps')
print(f'{"=" * 75}')

header = f'{"Metric":<20}'
for obj in objectives:
    header += f' {obj:>12}'
print(header)
print('-' * len(header))

for metric_key, label, fmt in key_metrics:
    if metric_key not in summary_df.columns:
        continue
    row = f'{label:<20}'
    for obj in objectives:
        subset = summary_df[summary_df['objective'] == obj][metric_key]
        val = subset.mean() if len(subset) > 0 else float('nan')
        if np.isfinite(val):
            row += f' {val:>12{fmt}}'
        else:
            row += f' {"N/A":>12}'
    print(row)

print(f'\nTerminal values (end of episode):')
terminal_keys = [
    ('terminal_worker_min',       'Floor Wealth'),
    ('terminal_agency_floor',     'Agency Floor'),
    ('terminal_all_gini',         'Gini'),
    ('terminal_horizon_index',    'Horizon Index'),
    ('terminal_trust_planner',    'Planner Trust'),
    ('terminal_trust_institutional','Inst. Trust'),
    ('terminal_tech_frontier',    'Tech Frontier'),
    ('terminal_tech_mean',        'Tech Mean'),
]
for metric_key, label in terminal_keys:
    if metric_key not in summary_df.columns:
        continue
    row = f'  {label:<18}'
    for obj in objectives:
        subset = summary_df[summary_df['objective'] == obj][metric_key]
        val = subset.mean() if len(subset) > 0 else float('nan')
        row += f' {val:>12.3f}' if np.isfinite(val) else f' {"N/A":>12}'
    print(row)
print()

---
## Step 7 -- Trajectory plots

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Load raw trajectories for seed=42
raw = {}
for obj in OBJECTIVES:
    path = f'{OUTPUT_DIR}/raw_data/{obj}_seed42.parquet'
    if os.path.exists(path):
        raw[obj] = pd.read_parquet(path)

COLORS = {'SUM_RAW': '#E74C3C', 'NASH_MIN': '#F39C12', 'TOPO_X': '#3498DB', 'TOPO_MIN': '#2ECC71'}

metrics_to_plot = [
    ('all_gini',            'Gini Coefficient',    (0, 1)),
    ('worker_min',          'Floor Wealth',        None),
    ('agency_floor',        'Agency Floor',        None),
    ('unemployment_rate',   'Unemployment Rate',   (0, 1)),
    ('horizon_index',       'Horizon Index',       (0, 1)),
    ('trust_planner',       'Planner Trust',       (0, 1)),
    ('trust_institutional', 'Institutional Trust', (0, 1)),
    ('tech_frontier',       'Tech Frontier',       None),
    ('mean_firm_floor',     'Firm SEVC Floor',     (0, 1)),
    ('epistemic_health',    'Epistemic Health',    (0, 1)),
    ('n_workers',           'Population',          None),
    ('total_production',    'Total Production',    None),
]

n_rows = (len(metrics_to_plot) + 2) // 3
fig, axes = plt.subplots(n_rows, 3, figsize=(18, 4.5 * n_rows))
axes = axes.flatten()

for i, (metric, title, ylim) in enumerate(metrics_to_plot):
    ax = axes[i]
    for obj, df in raw.items():
        if metric in df.columns:
            ax.plot(df['step'], df[metric], color=COLORS.get(obj, 'grey'),
                    lw=1.5, label=obj, alpha=0.85)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Step')
    if ylim:
        ax.set_ylim(ylim)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

# Hide unused axes
for j in range(len(metrics_to_plot), len(axes)):
    axes[j].set_visible(False)

fig.suptitle(f'Trajectory Comparison (seed=42, {N_STEPS} steps)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/trajectory_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {OUTPUT_DIR}/trajectory_comparison.png')

---
## Step 8 -- Statistical tests

In [ ]:
import analysis
importlib.reload(analysis)

try:
    analysis_results = analysis.run_analysis(summary_df, OUTPUT_DIR)
except Exception as e:
    print(f'Statistical analysis: {e}')
    print('(3 seeds per condition is thin for significance testing.')
    print(' Results are directional. Run more seeds for publication-grade stats.)')

---
## Step 9 -- Trust & innovation deep dive

In [ ]:
# Trust trajectory comparison
trust_metrics = ['trust_planner', 'trust_institutional', 'trust_firm_mean',
                 'trust_worker_mean', 'trust_news_mean', 'trust_bank_mean']

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
axes = axes.flatten()
for i, metric in enumerate(trust_metrics):
    ax = axes[i]
    for obj, df in raw.items():
        if metric in df.columns:
            ax.plot(df['step'], df[metric], color=COLORS.get(obj, 'grey'),
                    lw=1.5, label=obj, alpha=0.85)
    ax.set_title(metric.replace('trust_', '').replace('_', ' ').title() + ' Trust', fontsize=11)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
fig.suptitle('Trust Score Trajectories', fontsize=14)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/trust_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Innovation trajectory comparison
tech_metrics = ['tech_frontier', 'tech_mean', 'tech_gini', 'n_innovators']

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for i, metric in enumerate(tech_metrics):
    ax = axes[i]
    for obj, df in raw.items():
        if metric in df.columns:
            ax.plot(df['step'], df[metric], color=COLORS.get(obj, 'grey'),
                    lw=1.5, label=obj, alpha=0.85)
    ax.set_title(metric.replace('_', ' ').title(), fontsize=11)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
fig.suptitle('Innovation Trajectories', fontsize=14)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/innovation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Trust and innovation plots saved.')

---
## Step 10 -- Download results

In [ ]:
try:
    from google.colab import files
    import shutil
    zip_path = '/content/comparison_results'
    shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
    files.download(zip_path + '.zip')
    print('Download started.')
except ImportError:
    print(f'Results in: {OUTPUT_DIR}')
    !du -sh {OUTPUT_DIR}

---
## Interpreting results

### What to look for

**SUM_RAW vs everything else**: Does extraction still emerge with trust, innovation, and sustainable capitalism active but no sustainability constraint on the planner? If SUM_RAW still produces high Gini, poverty traps, and floor collapse, the objective function is the mechanism. If it doesn't, the structural systems (trust/innovation/SEVC) are doing the work.

**TOPO_X vs TOPO_MIN**: Does the min gate produce faster horizon index convergence? The multiplier gives proportional credit for partial sustainability. The min gate says sustainability IS the reward until it's fixed. Watch the horizon index trajectory: TOPO_MIN should stabilize faster but might have lower peak production.

**NASH_MIN**: Does the emergency brake on Nash welfare protect the floor better than a multiplier would? Compare floor wealth and agency floor between NASH_MIN and TOPO_X (which both use Nash-like bases but different HI integration).

**Trust dynamics**: Does planner trust diverge across conditions? SUM_RAW should produce lower planner trust (erratic policy, floor compression). Do low-trust firms struggle to hire (check firm trust vs employment)?

**Innovation**: Does tech frontier grow faster under some objectives? Innovation requires capital and stability. Objectives that stabilize the economy should produce more cumulative innovation.

### Caveats
- 3 seeds is thin for significance testing. Results are directional.
- The planner's ES optimizer needs ~500-800 steps to converge. Early behavior is noisy.
- Animation files can be large. The subsample=3 setting keeps them manageable.
- The `_collect_animation = False` flag on non-animated seeds saves ~30% memory.

### Running more seeds
Change `SEEDS = [42, 137, 2024]` to `SEEDS = list(range(10))` in Step 5 for 10 seeds per condition (40 episodes total). On H100 this adds ~40 min.